In [3]:
%pip install qiskit torch numpy pandas

   ---------------------------------------- 0.0/212.5 MB ? eta -:--:--
    --------------------------------------- 2.9/212.5 MB 14.0 MB/s eta 0:00:15
   - -------------------------------------- 6.6/212.5 MB 16.1 MB/s eta 0:00:13
   - -------------------------------------- 10.5/212.5 MB 17.2 MB/s eta 0:00:12
   -- ------------------------------------- 14.2/212.5 MB 17.1 MB/s eta 0:00:12
   --- ------------------------------------ 17.8/212.5 MB 17.6 MB/s eta 0:00:12
   ---- ----------------------------------- 21.5/212.5 MB 17.7 MB/s eta 0:00:11
   ---- ----------------------------------- 24.9/212.5 MB 17.5 MB/s eta 0:00:11
   ----- ---------------------------------- 27.0/212.5 MB 16.5 MB/s eta 0:00:12
   ----- ---------------------------------- 29.1/212.5 MB 15.8 MB/s eta 0:00:12
   ----- ---------------------------------- 31.7/212.5 MB 15.6 MB/s eta 0:00:12
   ------ --------------------------------- 34.3/212.5 MB 15.4 MB/s eta 0:00:12
   ------ --------------------------------- 36.7/21

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pennylane-qiskit 0.40.1 requires sympy<1.13, but you have sympy 1.14.0 which is incompatible.
qiskit-addon-cutting 0.10.0 requires qiskit<3,>=1.3.1, but you have qiskit 1.2.4 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import torch
import torch.nn as nn
from torch.autograd import Function
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp

#Prep Training Data (Breast Cancer Wisconsin)

#Load the data file
df = pd.read_csv("data\\wdbc.data", header=None)

#Assign column names
columns = ['id', 'diagnosis'] + [f'feature_{i}' for i in range(1, 31)]
df.columns = columns

#Drop the ID column
df = df.drop(columns=['id'])

#Encode diagnosis: M = 1, B = 0
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

#Convert to numpy arrays
X = df.drop(columns=['diagnosis']).values.astype(np.float32)
Y = df['diagnosis'].values.astype(np.float32).reshape(-1, 1)

#Normalize features manually (z-score)
mean = X.mean(axis=0)
std = X.std(axis=0)
X = (X - mean) / std

#Convert to torch tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
Y_tensor = torch.tensor(Y, dtype=torch.float32)

#Manual train/test split (80/20)
num_samples = X_tensor.shape[0]
indices = torch.randperm(num_samples)

split_idx = int(num_samples * 0.8)
train_indices = indices[:split_idx]
test_indices = indices[split_idx:]

X_train = X_tensor[train_indices]
Y_train = Y_tensor[train_indices]
X_test = X_tensor[test_indices]
Y_test = Y_tensor[test_indices]

#VQA Circuit
n_qubits = 2
params = ParameterVector('theta', length=4)

def create_vqa_circuit(input_data, weights):
    qc = QuantumCircuit(n_qubits)
    print("Make changes here")
    return qc

#Qiskit StatevectorEstimator primitive
estimator = StatevectorEstimator()
observables = [SparsePauliOp("ZI")]

#PyTorch Custom Autograd Function For VQA Layer
class VQALayerFunction(Function):
    @staticmethod
    def forward(ctx, input_tensor, weights):
        input_vals = input_tensor.detach().numpy()
        weight_vals = weights.detach().numpy()
        ctx.save_for_backward(input_tensor, weights)

        qc = create_vqa_circuit(input_vals, weight_vals)
        job = estimator.run([(qc, observables)])
        expval = job.result()[0].data.evs

        return torch.tensor([expval], dtype=torch.float32)

    @staticmethod
    def backward(ctx, grad_output):
        input_tensor, weights = ctx.saved_tensors
        input_vals = input_tensor.detach().numpy()
        weight_vals = weights.detach().numpy()
        shift = np.pi / 2
        grads = np.zeros_like(weights.detach().numpy(), dtype=np.float32)

        for i in range(len(weight_vals)):
            print("Make changes here")

        grads_tensor = torch.tensor(grads, dtype=torch.float32)
        return None, (grad_output.view(-1)[0] * grads_tensor).view(-1)

#Quantum Layer as PyTorch Module
class VQALayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(4)) #4 trainable parameters for the circuit

    def forward(self, x):
        return torch.stack([VQALayerFunction.apply(x[i], self.weights) for i in range(x.size(0))]).view(-1, 1)

#Full Hybrid Model
class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.classical = nn.Linear(X_tensor.shape[1], 2) #Classic preprocessing layer
        self.quantum = VQALayer() #VQA layer
        self.output = nn.Linear(1, 1) #Final classical layer

    def forward(self, x):
        x = self.classical(x)
        x = torch.tanh(x)  #Activation before quantum layer
        x = self.quantum(x)
        x = self.output(x)
        return torch.sigmoid(x)

model = HybridModel()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.BCELoss()

#Training Loop
for epoch in range(50):
    optimizer.zero_grad()
    preds = model(X_train)
    loss = loss_fn(preds, Y_train)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        acc = ((preds > 0.5).float() == Y_train).float().mean()
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f} | Accuracy: {acc.item()*100:.2f}%")

#Implement Testing Loop:


Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make changes here
Make chang

In [8]:
import torch
import torch.nn as nn
from torch.autograd import Function
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp

#Prep Training Data (Breast Cancer Wisconsin)

#Load the data file
df = pd.read_csv("data\\wdbc.data", header=None)

#Assign column names
columns = ['id', 'diagnosis'] + [f'feature_{i}' for i in range(1, 31)]
df.columns = columns

#Drop the ID column
df = df.drop(columns=['id'])

#Encode diagnosis: M = 1, B = 0
df['diagnosis'] = df['diagnosis'].map({'M': 1, 'B': 0})

#Convert to numpy arrays
X = df.drop(columns=['diagnosis']).values.astype(np.float32)
Y = df['diagnosis'].values.astype(np.float32).reshape(-1, 1)

#Normalize features manually (z-score)
mean = X.mean(axis=0)
std = X.std(axis=0)
X = (X - mean) / std

#Convert to torch tensors
X_tensor = torch.tensor(X, dtype=torch.float32)
Y_tensor = torch.tensor(Y, dtype=torch.float32)

#Manual train/test split (80/20)
num_samples = X_tensor.shape[0]
indices = torch.randperm(num_samples)

split_idx = int(num_samples * 0.8)
train_indices = indices[:split_idx]
test_indices = indices[split_idx:]

X_train = X_tensor[train_indices]
Y_train = Y_tensor[train_indices]
X_test = X_tensor[test_indices]
Y_test = Y_tensor[test_indices]

#VQA Circuit
n_qubits = 2
params = ParameterVector('theta', length=4)

# --- 1. Variational circuit -------------------------------------------------
n_qubits = 2          # keep this in one place

def create_vqa_circuit(inputs, weights):
    """
    inputs  : 1-D numpy array of length 2  (classical features)
    weights : 1-D numpy array of length 4  (trainable parameters)
    """
    qc = QuantumCircuit(n_qubits)

    # --- feature encoding (one RY per qubit) -------------
    for q, x in enumerate(inputs):
        qc.ry(float(x), q)

    # --- simple entanglement -----------------------------
    qc.cx(0, 1)

    # --- variational layer -------------------------------
    qc.ry(float(weights[0]), 0)
    qc.rz(float(weights[1]), 0)
    qc.ry(float(weights[2]), 1)
    qc.rz(float(weights[3]), 1)

    return qc


#Qiskit StatevectorEstimator primitive
estimator = StatevectorEstimator()
observables = [SparsePauliOp("ZI")]

#PyTorch Custom Autograd Function For VQA Layer
# --- 2. Autograd function with parameter-shift -----------

class VQALayerFunction(Function):
    @staticmethod
    def forward(ctx, input_tensor, weights):
        # save tensors for backward
        ctx.save_for_backward(input_tensor, weights)

        # run circuit once
        expval = StatevectorEstimator().run(
            [(create_vqa_circuit(input_tensor.numpy(), weights.detach().numpy()),
              SparsePauliOp("ZI"))]
        ).result()[0].data.evs

        # return a scalar tensor
        return torch.tensor([expval], dtype=torch.float32, device=input_tensor.device)

    @staticmethod
    def backward(ctx, grad_output):
        input_tensor, weights = ctx.saved_tensors
        x   = input_tensor.detach().numpy()
        th  = weights.detach().numpy()
        sh  = np.pi / 2                                   # parameter-shift amount

        grads = np.zeros_like(th, dtype=np.float32)
        est   = StatevectorEstimator()

        for i in range(len(th)):
            th_plus, th_minus = th.copy(), th.copy()
            th_plus[i]  += sh
            th_minus[i] -= sh

            # expectation at θ+ and θ-
            exp_plus  = est.run([(create_vqa_circuit(x, th_plus),  SparsePauliOp("ZI"))]
                               ).result()[0].data.evs
            exp_minus = est.run([(create_vqa_circuit(x, th_minus), SparsePauliOp("ZI"))]
                               ).result()[0].data.evs

            # parameter-shift derivative (generator eigenvalues ±½ ⇒ prefactor 0.5)
            grads[i] = 0.5 * (exp_plus - exp_minus)

        # convert to tensor and apply chain rule
        grad_weights = grad_output.view(-1)[0] * torch.tensor(grads,
                                                              dtype=torch.float32,
                                                              device=weights.device)

        return None, grad_weights   # No gradient w.r.t. the classical input


#Quantum Layer as PyTorch Module
class VQALayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(4)) #4 trainable parameters for the circuit

    def forward(self, x):
        return torch.stack([VQALayerFunction.apply(x[i], self.weights) for i in range(x.size(0))]).view(-1, 1)

#Full Hybrid Model
class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.classical = nn.Linear(X_tensor.shape[1], 2) #Classic preprocessing layer
        self.quantum = VQALayer() #VQA layer
        self.output = nn.Linear(1, 1) #Final classical layer

    def forward(self, x):
        x = self.classical(x)
        x = torch.tanh(x)  #Activation before quantum layer
        x = self.quantum(x)
        x = self.output(x)
        return torch.sigmoid(x)

model = HybridModel()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
loss_fn = nn.BCELoss()

#Training Loop
for epoch in range(50):
    optimizer.zero_grad()
    preds = model(X_train)
    loss = loss_fn(preds, Y_train)
    loss.backward()
    optimizer.step()
    with torch.no_grad():
        acc = ((preds > 0.5).float() == Y_train).float().mean()
        print(f"Epoch {epoch+1} | Loss: {loss.item():.4f} | Accuracy: {acc.item()*100:.2f}%")

#Implement Testing Loop:


TypeError: len() of unsized object